# Bayes' rule

_Probability & Statistics_

**Flip a conditional probability around. The secret weapon for tests and beliefs.**

Sometimes you know $P(B \mid A)$ but really want $P(A \mid B)$.

     Bayes' rule flips one into the other.

---

This notebook builds the classic **medical-test** example one piece at a time: first the exact arithmetic, then a large simulation to confirm it, then a picture of how one positive test reshapes your belief. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy + SciPy

We work through the textbook **rare-disease test**: a disease that affects 1 in 1000 people, a test that is 99% sensitive, and a 1% false-positive rate. We build it in three steps: (1) the exact posterior from Bayes' rule, (2) a simulation that should land on the same number, (3) comparing the two.

### Step 1 — Compute the posterior with Bayes' rule

Start with the three numbers that define the problem: the **prior** $P(\text{sick})$, the **sensitivity** $P(+\mid\text{sick})$, and the **false-positive rate** $P(+\mid\text{healthy})$.

To flip $P(+\mid\text{sick})$ into $P(\text{sick}\mid +)$ we need the total chance of a positive test — the law of total probability adds the true positives and the false positives. Bayes' rule then divides the true-positive mass by that total. The posterior is much smaller than your intuition expects, because the disease is so rare that most positives come from healthy people.

In [ ]:
prior = 0.001        # P(sick): the disease is rare, 1 in 1000
sens = 0.99          # P(+ | sick): the test catches 99% of sick people
fpr = 0.01           # P(+ | healthy): 1% of healthy people test positive anyway

# Law of total probability: a positive test comes from a true positive or a false positive.
true_positive = sens * prior
false_positive = fpr * (1 - prior)
p_pos = true_positive + false_positive

# Bayes' rule: of all the positives, what fraction are actually sick?
post = true_positive / p_pos
print("P(sick | +) =", round(post, 4))

### Step 2 — Confirm it by simulating a population

Bayes' rule is just arithmetic, so let's sanity-check it against a brute-force simulation. We draw a huge population, randomly mark each person sick with probability `prior`, then give each person a test whose outcome depends on whether they are sick.

Among everyone who tested positive, we measure the fraction who are truly sick — that empirical fraction *is* $P(\text{sick}\mid +)$.

In [ ]:
rng = np.random.default_rng(0)
n = 2_000_000

# Randomly decide who is actually sick (each person sick with probability = prior).
sick = rng.random(n) < prior

# Test everyone: sick people test positive with prob sens, healthy with prob fpr.
pos = np.where(sick, rng.random(n) < sens, rng.random(n) < fpr)

# Of all the positive tests, what fraction belong to people who are actually sick?
sim_post = np.mean(sick[pos])
print("sim P(sick | +) =", round(sim_post, 4))

### Step 3 — Compare exact vs simulated

The closed-form posterior from Step 1 and the simulated fraction from Step 2 should agree to a few decimal places. Any tiny gap is just sampling noise from the finite population — it shrinks as `n` grows.

In [ ]:
print("exact   P(sick | +) =", round(post, 4))
print("simulated P(sick | +) =", round(sim_post, 4))

## Visualize the data & results

_After a positive test, how does belief shift from prior to posterior?_

### Step 1 — Recompute the prior and posterior

The plot needs two numbers: your belief *before* the test (the prior) and your belief *after* a positive test (the posterior). We recompute them here with the same Bayes' rule so this cell stands on its own.

In [ ]:
# Rare disease, 99% sensitive test, 1% false positive rate.
prior = 0.001
sens = 0.99
fpr = 0.01

# Bayes' rule: posterior P(sick | +).
p_pos = sens * prior + fpr * (1 - prior)
post = sens * prior / p_pos
vals = [prior, post]

### Step 2 — Plot the belief shift

A simple bar chart puts the prior next to the posterior. Even though the posterior is still well below 50%, notice how dramatically one positive test moves your belief — from 0.1% up to a few tens of percent, a jump of two orders of magnitude.

In [ ]:
labels = ['prior P(sick)', 'posterior P(sick | +)']
colors = ['#9aa7b4', '#ff7b72']

plt.bar(labels, vals, color=colors)
plt.title('Prior P(sick) vs posterior P(sick | +)')
plt.ylabel('probability')
plt.show()